In [ ]:
import sys; sys.path.append("..")
import numpy as np, pandas as pd
from scipy.stats import poisson
from sklearn.linear_model import PoissonRegressor

from src.splits import split
from src.evaluation.metrics import evaluate, confusion
from src.evaluation.registro import guardar_resultado

df = pd.read_csv("../data/processed/features.csv", parse_dates=["date"])
train, val = split(df)                       # nunca usa la temporada de prueba

X_tr = train[["elo_diff"]] / 100
modelo_local = PoissonRegressor(alpha=0).fit(X_tr, train["home_goals"])
modelo_visit = PoissonRegressor(alpha=0).fit(X_tr, train["away_goals"])

print("Local:     intercepto", modelo_local.intercept_.round(3), "coef", modelo_local.coef_.round(3))
print("Visitante: intercepto", modelo_visit.intercept_.round(3), "coef", modelo_visit.coef_.round(3))

print("\nGoles esperados según la diferencia de Elo (en cientos):")
for d in [0, 1, 3]:
    x = pd.DataFrame({"elo_diff": [d]})
    print(f"  elo_diff={d*100:>3}: local {modelo_local.predict(x)[0]:.2f}  visitante {modelo_visit.predict(x)[0]:.2f}")

In [ ]:
def probs_hda(lam_local, lam_visit, max_goles=10):
    """Probabilidades [H, D, A] a partir de los goles esperados de cada equipo."""
    k = np.arange(max_goles + 1)
    matriz = np.outer(poisson.pmf(k, lam_local), poisson.pmf(k, lam_visit))   # filas: goles local; columnas: goles visitante
    p = np.array([np.tril(matriz, -1).sum(), np.trace(matriz), np.triu(matriz, 1).sum()])
    return p / p.sum()          # renormaliza lo poco que cae fuera de 10 goles

# Comprobación a mano: un partido parejo
print("Parejo (1.3 vs 1.3):", probs_hda(1.3, 1.3).round(3))
print("Local favorito (2.4 vs 0.7):", probs_hda(2.4, 0.7).round(3))

# Predicción en validación (2023-24)
X_val = val[["elo_diff"]] / 100
lam_l = modelo_local.predict(X_val)
lam_v = modelo_visit.predict(X_val)
P = np.array([probs_hda(a, b) for a, b in zip(lam_l, lam_v)])

res = evaluate(val["result"], P, "poisson_elo")
print(res)
print("Probabilidad media H/D/A:", P.mean(axis=0).round(3))
print("Rango de P(empate):", P[:, 1].min().round(3), "-", P[:, 1].max().round(3))
print(confusion(val["result"], P))